# MiniZinc (constraint programming)

**Domain:** Symbolic AI & Logic  ·  **recommended addition**  ·  **runnable:** yes

A refresher on MiniZinc: a high-level, solver-agnostic language for **constraint satisfaction and optimization**. You describe *what* a valid solution looks like; the solver figures out *how* to find one.

## 1. What & Why

**MiniZinc is a modeling language for combinatorial problems**, not a solver itself. You write a declarative model — decision variables, their domains, and the constraints that relate them — and MiniZinc compiles it down to **FlatZinc**, a low-level form that many backend solvers understand (Gecode, Chuffed, OR-Tools CP-SAT, CPLEX, Gurobi, ...). The same `.mzn` model can be thrown at a constraint-propagation solver, a SAT/lazy-clause solver, or a MIP solver without rewriting it.

**The problem it solves.** A huge class of real problems are "assign discrete choices subject to rules, optionally optimizing something": timetabling, rostering, vehicle routing, packing/cutting, scheduling, configuration, puzzle solving. Writing a bespoke backtracking search for each is slow to build and slower to run. MiniZinc lets you state the problem in a few dozen lines and hand the *search* to a battle-tested solver with global constraints, propagation, and learned-clause search.

**When to reach for it:**
- You can describe a solution by *constraints* ("no two queens share a diagonal", "each nurse works ≤ 5 shifts") rather than by an algorithm.
- The problem is discrete/combinatorial and NP-hard-ish — exhaustive search is hopeless but constraint propagation prunes hard.
- You want to prototype a model fast and compare solvers, or you need a *proven-optimal* answer, not a heuristic guess.

**When *not* to:** continuous nonlinear optimization (use a numerical optimizer), pure linear programs at massive scale (call a MIP solver directly), or anything where a greedy/heuristic answer is good enough and latency is critical.

## 2. Mental Model

Think of MiniZinc as a **spreadsheet of unknowns with rules attached**. You declare cells (decision variables) and the allowed values for each (domains), then write rules that any valid filling must obey. You never tell it the order to try values — the solver does *constraint propagation* (each rule shrinks the others' domains) interleaved with *search* (guess a value, propagate, backtrack on failure).

```
  model.mzn  ──(mzn2fzn compile)──►  model.fzn  ──►  [ solver ]  ──►  solution(s)
  vars + domains + constraints        flat, solver-       Gecode / Chuffed /
  + solve goal                        independent IR      OR-Tools / CPLEX ...
```

The key inversion from normal programming: **you separate *model* (the what) from *solver* (the how).** Your job is to write the tightest, most propagation-friendly description of feasibility. The solver's job is the search. A good *global constraint* like `alldifferent` isn't just sugar — it carries a specialized propagation algorithm that prunes far more than the equivalent pile of pairwise `!=` constraints.

## 3. Key Concepts

- **Decision variable (`var`)** vs **parameter (`par`)** — a `var` is an unknown the solver chooses (`var 1..9: x`); a parameter is fixed input data (`int: n`). Everything not marked `var` is a parameter.
- **Domain** — the set of allowed values for a variable (`var 1..n`, `var 0..1`, `var bool`, `set of int`). Tighter domains = more propagation.
- **Constraint** — a Boolean relation that must hold: `constraint sum(x) <= cap;`. Constraints are *relations*, not assignments — order doesn't matter.
- **Solve goal** — `solve satisfy;` (find any feasible solution), `solve minimize expr;`, or `solve maximize expr;`.
- **Global constraints** — named, high-level constraints with strong dedicated propagators: `alldifferent`, `cumulative` (scheduling), `circuit` (routing/TSP), `bin_packing`, `table`. Pull them in with `include "alldifferent.mzn";`.
- **Comprehensions & generators** — array/constraint comprehensions like `[q[i] + i | i in 1..n]` and `forall(i in S)(...)` / `sum(i in S)(...)` build constraints over index sets compactly.
- **Model / data separation** — keep the `.mzn` generic and pass instance data via a `.dzn` data file or by binding parameters at runtime, so one model solves many instances.
- **FlatZinc** — the compiled intermediate the solver actually runs; usually invisible, but where solver-specific behavior lives.
- **Search annotations** — optional hints like `solve :: int_search(q, first_fail, indomain_min) satisfy;` to steer variable/value ordering when the default search is slow.

## 4. Setup

MiniZinc has two pieces: the **MiniZinc toolchain** (the `minizinc` compiler + bundled solvers like Gecode/Chuffed) and the **Python interface** (`pip install minizinc`), which shells out to that toolchain.

- **Toolchain (system binary, ~one large download):** install the MiniZinc IDE bundle from [minizinc.org/software.html](https://www.minizinc.org/software.html), or `conda install -c conda-forge minizinc`, or your OS package manager. This ships Gecode and Chuffed so you can solve immediately.
- **Python package:** `%pip install minizinc` — a thin driver that compiles models and parses solver output. It needs the binary above on `PATH`.

```bash
# one-time, outside the notebook:
conda install -c conda-forge minizinc      # gets the compiler + Gecode + Chuffed
pip install minizinc                        # the Python bindings
```

Because the toolchain is a sizable platform install, the code cells below **detect whether it's present** and fall back to a tiny pure-Python solver for the *same* problem so the notebook executes end-to-end either way. The MiniZinc model strings are the real teaching artifact; the production call shape is in `solve_mzn` below.

In [ ]:
# %pip install minizinc   # uncomment on a fresh kernel (also needs the MiniZinc toolchain on PATH)
import shutil
import importlib.util

MZN_BIN = shutil.which("minizinc")
HAS_PYLIB = importlib.util.find_spec("minizinc") is not None
print("minizinc binary :", MZN_BIN or "NOT FOUND  (install from minizinc.org)")
print("python package  :", "available" if HAS_PYLIB else "not installed (pip install minizinc)")


def solve_mzn(model_str, solver="gecode", **data):
    """Run a MiniZinc model and return the solver Result, or None if the
    toolchain is unavailable. This is the real call shape you'd use in code."""
    if not (MZN_BIN and HAS_PYLIB):
        return None
    import minizinc
    model = minizinc.Model()
    model.add_string(model_str)
    inst = minizinc.Instance(minizinc.Solver.lookup(solver), model)
    for key, value in data.items():
        inst[key] = value
    return inst.solve()


print("\nsolve_mzn ready — will run real MiniZinc when the toolchain is present.")

## 5. Worked Examples

Two classics: **N-Queens** (a pure satisfaction problem showing `alldifferent` and the diagonal trick) and **0/1 knapsack** (an optimization problem with `maximize`). Each prints the MiniZinc model, runs it if the toolchain is installed, and otherwise verifies the answer with a small Python solver so you see real output.

### Example 1 — N-Queens (satisfaction)

Place `n` queens on an `n×n` board so none attack each other. The model: one variable per row holding that queen's column. Columns must differ (`alldifferent(q)`); diagonals differ when `q[i]+i` and `q[i]-i` are each all-distinct — a textbook MiniZinc idiom.

In [ ]:
nqueens = """
include "alldifferent.mzn";
int: n;
array[1..n] of var 1..n: q;                      % q[i] = column of the queen in row i
constraint alldifferent(q);                       % distinct columns
constraint alldifferent([q[i] + i | i in 1..n]);  % distinct '/' diagonals
constraint alldifferent([q[i] - i | i in 1..n]);  % distinct '\\' diagonals
solve satisfy;
"""
print(nqueens)

res = solve_mzn(nqueens, n=8)
if res is not None:
    cols = res["q"]
    print("MiniZinc solution (column per row):", cols)
else:
    # Toolchain absent here: verify the model's logic with a tiny backtracker.
    def queens(n, placed=()):
        row = len(placed)
        if row == n:
            return placed
        for col in range(1, n + 1):
            if all(col != c and abs(col - c) != row - r
                   for r, c in enumerate(placed)):
                sol = queens(n, placed + (col,))
                if sol:
                    return sol
        return None
    cols = queens(8)
    print("(toolchain absent) verified n=8 solution:", cols)

# pretty-print the board
for c in cols:
    print(" ".join("Q" if j == c else "." for j in range(1, len(cols) + 1)))

### Example 2 — 0/1 Knapsack (optimization)

Pick a subset of items to **maximize profit** without exceeding a weight `capacity`. Note `solve maximize` and how the model stays generic — all instance data (`profit`, `weight`, `capacity`) are parameters bound at solve time, so the same `.mzn` handles any knapsack.

In [ ]:
knapsack = """
int: n;
set of int: ITEM = 1..n;
array[ITEM] of int: profit;
array[ITEM] of int: weight;
int: capacity;
array[ITEM] of var 0..1: take;                    % 1 = item is in the knapsack
constraint sum(i in ITEM)(weight[i] * take[i]) <= capacity;
solve maximize sum(i in ITEM)(profit[i] * take[i]);
"""
print(knapsack)

profit, weight, capacity = [60, 100, 120], [10, 20, 30], 50
res = solve_mzn(knapsack, n=len(profit), profit=profit, weight=weight, capacity=capacity)
if res is not None:
    print("take :", res["take"], "  max profit :", res.objective)
else:
    # Toolchain absent: brute-force the 2^n subsets (n is tiny) to confirm the optimum.
    from itertools import product
    best_profit, best_take = max(
        (sum(p for p, t in zip(profit, take) if t), take)
        for take in product([0, 1], repeat=len(profit))
        if sum(w for w, t in zip(weight, take) if t) <= capacity
    )
    print("(toolchain absent) take :", list(best_take), "  max profit :", best_profit)

## 6. Gotchas & Pitfalls

- **Pairwise `!=` instead of a global constraint.** Writing `forall(i,j where i<j)(q[i] != q[j])` works but propagates weakly; `alldifferent(q)` uses a matching-based propagator that prunes far more and scales. Always prefer the global form when one exists.
- **Forgetting the `include`.** Global constraints live in library files — `alldifferent`, `cumulative`, `circuit`, etc. each need `include "<name>.mzn";` or you get an "undefined identifier" error.
- **`var` vs parameter confusion.** Indexing an array with a `var` index, or using a `var` where the language needs a fixed value (e.g. an array size), triggers cryptic flattening errors. Keep structure fixed at parameters; let only the unknowns be `var`.
- **Unbounded domains.** `var int: x` with no bounds can blow up flattening or make the solver flail. Always bound domains as tightly as the problem allows.
- **`satisfy` returns *a* solution, not *all*.** To enumerate, use the solver's all-solutions mode (`-a` / `Instance.solve(all_solutions=True)`); to count or rank, model it explicitly. Don't assume the first solution is special.
- **Optimization without symmetry breaking is slow.** Symmetric solutions (e.g. interchangeable workers) multiply the search space. Add symmetry-breaking constraints (`lex_lesseq`, orderings) — often a 10–100× speedup.
- **Float constraints quietly switch worlds.** Mixing `var float` pulls in a different (MIP-style) solving path; many global constraints are integer-only. Keep models integer unless you truly need continuous values.
- **Solver choice matters a lot.** Gecode (CP) is a great default; Chuffed (lazy clause generation) often crushes hard scheduling/optimization; CP-SAT/OR-Tools is strong on optimization. Same model, very different runtimes — benchmark before optimizing the model.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-off vs MiniZinc |
|---|---|---|
| **MiniZinc** | Discrete CSP/optimization you can state as constraints; comparing solvers; teaching/prototyping | Solver-agnostic, concise, global constraints; but a separate toolchain, and not for continuous/nonlinear math |
| **OR-Tools CP-SAT (Python)** | Production scheduling/routing at scale, all-in-Python pipelines | Tighter Python integration and a superb solver, but you hand-build the model in API calls — less portable across solvers |
| **Z3 / SMT** | Verification, logic with rich theories (arrays, bitvectors), satisfiability of formulas | More general logical reasoning; weaker at large optimization and lacks CP global-constraint propagators ([see `z3`](z3.ipynb)) |
| **MIP solvers (Gurobi/CPLEX, PuLP)** | Large *linear* integer programs | Unmatched on linear structure; awkward for highly combinatorial/logical constraints, and you linearize everything by hand |
| **Hand-written backtracking / local search** | Tiny problems, or when you need a custom heuristic | Full control and zero deps, but you reimplement propagation and search — slow to build, easy to get wrong |
| **Prolog / answer-set programming** | Relational logic, rule inference | Great for logical deduction; less natural for numeric optimization objectives |

**Rule of thumb:** if you can phrase it as "choose discrete values subject to these rules (optionally optimizing one)", start in MiniZinc — it's the fastest way to a correct model and lets you swap solvers freely. Reach for OR-Tools when you've outgrown the toolchain and want everything in one Python process, for MIP when the problem is essentially linear, and for [Z3](z3.ipynb) when you need general logical reasoning over rich theories.

## 8. Resources

- **Official handbook & language reference** — [docs.minizinc.dev/en/stable/](https://docs.minizinc.dev/en/stable/) — the canonical docs, including the tutorial.
- **"Modeling with MiniZinc" / Coursera "Basic & Advanced Modeling for Discrete Optimization"** — [coursera.org/learn/basic-modeling](https://www.coursera.org/learn/basic-modeling) — the definitive course from MiniZinc's authors.
- **Python interface docs** — [python.minizinc.dev/en/latest/](https://python.minizinc.dev/en/latest/) — the `minizinc` package used in `solve_mzn` above.
- **Downloads (toolchain + IDE + bundled solvers)** — [minizinc.org/software.html](https://www.minizinc.org/software.html).
- **The Global Constraint Catalogue** — [sofdem.github.io/gccat/](https://sofdem.github.io/gccat/) — reference for global constraints and what they mean.

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def propagate_alldifferent(domains):
    """Shrink the domains as far as the two rules allow, or None on failure."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE